# Importing Library

In [16]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom , RandomTranslation , RandomContrast , RandomBrightness
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.regularizers import l2

# Importing Dataset

In [17]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Found 6799 files belonging to 3 classes.
Using 5440 files for training.
Found 6799 files belonging to 3 classes.
Using 1359 files for validation.
Found 2278 files belonging to 3 classes.


# Data Preprocessing

In [ ]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(factor=(-0.025, 0.025)),
    RandomTranslation(height_factor=0.05, width_factor=0.05),
    RandomZoom(height_factor=(-0.05, 0.05), width_factor=(-0.05, 0.05)),
    RandomContrast(factor=0.1),
    RandomFlip(mode="horizontal"),
    RandomBrightness(factor=0.1)
], name="data_augmentation")

def cutmix_sparse(batch_x, batch_y, alpha=1.0):
    batch_size = tf.shape(batch_x)[0]

    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_x = tf.gather(batch_x, indices)
    shuffled_y = tf.gather(batch_y, indices)

    lam = tf.random.uniform([], 0, 1)

    H, W = IMG_SIZE
    r_x = tf.cast(W * tf.random.uniform([], 0, 1), tf.int32)
    r_y = tf.cast(H * tf.random.uniform([], 0, 1), tf.int32)
    r_w = tf.cast(W * tf.math.sqrt(1 - lam), tf.int32)
    r_h = tf.cast(H * tf.math.sqrt(1 - lam), tf.int32)

    x1 = tf.clip_by_value(r_x - r_w // 2, 0, W)
    y1 = tf.clip_by_value(r_y - r_h // 2, 0, H)
    x2 = tf.clip_by_value(r_x + r_w // 2, 0, W)
    y2 = tf.clip_by_value(r_y + r_h // 2, 0, H)

    mask = tf.ones([y2 - y1, x2 - x1, 3])
    pad_top = y1
    pad_left = x1
    pad_bottom = H - y2
    pad_right = W - x2
    mask = tf.pad(mask, [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]])
    mask = 1 - mask

    mixed_x = batch_x * mask + shuffled_x * (1 - mask)

    mixed_y = tf.where(lam > 0.5, batch_y, shuffled_y)

    return mixed_x, mixed_y


train_ds_augmented = train_ds.map(
    lambda x, y: cutmix_sparse(x, y), num_parallel_calls=AUTOTUNE
)


# Model Creation

In [30]:
def build_model(input_shape=IMG_SIZE + (3,), num_classes=3):
    base_model = ResNet50(
        include_top=False,
        input_shape=input_shape,
        weights="imagenet"
    )
    base_model.trainable = False 

    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation="softmax", kernel_regularizer=l2(1e-4))(x)

    model = Model(inputs, outputs, name="Human_Emotion_Detection_Model")

    lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=1000,
        t_mul=2.0,
        m_mul=0.8,
        alpha=1e-6
    )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )
    return model, base_model

model, base_model = build_model()
model.summary()

Model: "Human_Emotion_Detection_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_18 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │         6,147 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,593,859 (90.00 MB)

 Trainable params: 6,147 (24.01 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

# Training The Model

In [31]:
history1 = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    ]
)

Epoch 1/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 23s 50ms/step - accuracy: 0.5202 - loss: 1.0355 - val_accuracy: 0.6689 - val_loss: 0.7192
Epoch 2/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - accuracy: 0.5759 - loss: 0.9205 - val_accuracy: 0.7079 - val_loss: 0.6898
Epoch 3/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.5985 - loss: 0.8775 - val_accuracy: 0.6939 - val_loss: 0.6939
Epoch 4/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - accuracy: 0.5877 - loss: 0.9167 - val_accuracy: 0.6762 - val_loss: 0.7537
Epoch 5/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 13s 38ms/step - accuracy: 0.5996 - loss: 0.8967 - val_accuracy: 0.7395 - val_loss: 0.6538


# Fine Tuning

## Total Layers of The Base Model

In [32]:
print("Base model has", len(base_model.layers), "layers.")

Base model has 175 layers.


## Tuning Process

In [33]:
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True


model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)

history2 = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        ModelCheckpoint("best_model.keras", save_best_only=True, monitor="val_loss"),
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

Epoch 1/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 47s 85ms/step - accuracy: 0.5807 - loss: 0.9112 - val_accuracy: 0.7594 - val_loss: 0.5852 - learning_rate: 1.0000e-05
Epoch 2/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/step - accuracy: 0.6404 - loss: 0.8164 - val_accuracy: 0.7792 - val_loss: 0.5481 - learning_rate: 1.0000e-05
Epoch 3/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/step - accuracy: 0.6708 - loss: 0.7554 - val_accuracy: 0.7947 - val_loss: 0.4995 - learning_rate: 1.0000e-05
Epoch 4/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 26s 76ms/step - accuracy: 0.6928 - loss: 0.7287 - val_accuracy: 0.8050 - val_loss: 0.4803 - learning_rate: 1.0000e-05
Epoch 5/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 25s 73ms/step - accuracy: 0.7048 - loss: 0.7096 - val_accuracy: 0.8109 - val_loss: 0.4719 - learning_rate: 1.0000e-05
Epoch 6/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/step - accuracy: 0.7153 - loss: 0.6779 - val_accuracy: 0.8263 - val_loss: 0.4393 - learning_rate: 1.0000e-05
Epoch 7/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 25s 72ms/ste

# Evaluate

In [34]:
model.evaluate(test_ds)

143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.8442 - loss: 0.4290


[0.4289568066596985, 0.8441615700721741]